# Лабораторная работа № 4. Векторизация и broadcasting: финансовое моделирование

**Кейс:** как изменить скидочную политику и выполнить план продаж.
**Данные:** `sales` — продажи 4 регионов (строки) по 6 неделям (столбцы), `target` — недельный план, `cost_rate` — доля себестоимости в выручке региона.
**Ограничения:** в обязательных заданиях нет циклов `for`/`while`, Pandas и ручного перебора; только векторизованные операции NumPy, маски и агрегации.

In [1]:
import numpy as np
np.set_printoptions(suppress=True, precision=2, linewidth=140)

sales = np.array([
    [12000, 15000, 18000, 21000, 19000, 23000],
    [9000, 11000, 14000, 16000, 13000, 15000],
    [20000, 22000, 19000, 25000, 24000, 26000],
    [8000, 13000, 17000, 15000, 14000, 18000]
])
target = np.array([15000, 16000, 17000, 18000, 20000, 22000])
cost_rate = np.array([0.70, 0.75, 0.65, 0.80])
regions = np.array(["Регион 1", "Регион 2", "Регион 3", "Регион 4"])
print(sales.shape, target.shape, cost_rate.shape)

(4, 6) (6,) (4,)


Формы: `sales` — `(4, 6)`, `target` — `(6,)`, `cost_rate` — `(4,)`. Деньги условные (ден. ед.).

## Задание 1. Выполнение плана

`sales - target` — это broadcasting: массив `target` формы `(6,)` растягивается по строкам, поэтому каждая неделя сравнивается с планом своей недели. Процент выполнения = `sales / target * 100`.

In [2]:
deviation = sales - target
met = sales >= target
completion = sales / target * 100
print("Отклонение от плана:\n", deviation)
print("План выполнен:\n", met)
print("Выполнение плана, %:\n", completion)
print("Недель с выполненным планом по регионам:", met.sum(axis=1))

Отклонение от плана:
 [[-3000 -1000  1000  3000 -1000  1000]
 [-6000 -5000 -3000 -2000 -7000 -7000]
 [ 5000  6000  2000  7000  4000  4000]
 [-7000 -3000     0 -3000 -6000 -4000]]
План выполнен:
 [[False False  True  True False  True]
 [False False False False False False]
 [ True  True  True  True  True  True]
 [False False  True False False False]]
Выполнение плана, %:
 [[ 80.    93.75 105.88 116.67  95.   104.55]
 [ 60.    68.75  82.35  88.89  65.    68.18]
 [133.33 137.5  111.76 138.89 120.   118.18]
 [ 53.33  81.25 100.    83.33  70.    81.82]]
Недель с выполненным планом по регионам: [3 0 6 1]


**Вывод.** План выполнен в 3 из 6 недель у Региона 1, в 0 из 6 у Региона 2, во всех 6 у Региона 3 и лишь в 1 неделе (3-й, ровно 100%) у Региона 4. Регион 3 стабильно перевыполняет план (112–139%), Регион 2 не дотягивает ни разу (60–89%).

## Задание 2. Оценка регионов

Суммы и средние считаются по оси `axis=1` (по неделям). `keepdims=True` сохраняет форму `(4, 1)`, поэтому `sales - mean` корректно вычитает среднее каждого региона из всех его недель. Для сравнения стабильности абсолютного стандартного отклонения мало — регионы разного масштаба, поэтому использован коэффициент вариации (std / среднее).

In [3]:
total = sales.sum(axis=1)
mean = sales.mean(axis=1, keepdims=True)
dev_mean = sales - mean
std = sales.std(axis=1)
cv = std / mean.ravel() * 100
print("Выручка по регионам:", total)
print("Среднее в неделю:", mean.ravel())
print("Отклонение от среднего:\n", dev_mean)
print("Стд. отклонение:", std)
print("Коэф. вариации, %:", cv)
print("Самый стабильный:", regions[cv.argmin()], "| наибольшие колебания:", regions[cv.argmax()])

Выручка по регионам: [108000  78000 136000  85000]
Среднее в неделю: [18000.   13000.   22666.67 14166.67]
Отклонение от среднего:
 [[-6000.   -3000.       0.    3000.    1000.    5000.  ]
 [-4000.   -2000.    1000.    3000.       0.    2000.  ]
 [-2666.67  -666.67 -3666.67  2333.33  1333.33  3333.33]
 [-6166.67 -1166.67  2833.33   833.33  -166.67  3833.33]]
Стд. отклонение: [3651.48 2380.48 2560.38 3236.08]
Коэф. вариации, %: [20.29 18.31 11.3  22.84]
Самый стабильный: Регион 3 | наибольшие колебания: Регион 4


**Вывод.** Выручка: Регион 3 — 136 000, Регион 1 — 108 000, Регион 4 — 85 000, Регион 2 — 78 000. **Наиболее стабилен Регион 3** (коэффициент вариации 11,3%), **наибольшие колебания у Региона 4** (22,8%). По абсолютному std самый большой разброс у Региона 1 (3 651), но относительно его среднего уровня он меньше, чем у Региона 4.

## Задание 3. Финансовая модель

`cost_rate` имеет форму `(4,)`, а `sales` — `(4, 6)`. Напрямую они несовместимы (сравниваются хвостовые оси 6 и 4), поэтому `cost_rate[:, np.newaxis]` превращает его в `(4, 1)`. Тогда ось длины 1 растягивается до 6: ставка региона применяется ко всем его неделям. Себестоимость = `sales * cost_rate`, валовая прибыль = `sales - cost`.

In [4]:
cost = sales * cost_rate[:, np.newaxis]
gross = sales - cost
print(cost_rate.shape, cost_rate[:, np.newaxis].shape, sales.shape, cost.shape)
print("Себестоимость:\n", cost)
print("Валовая прибыль:\n", gross)
print("Прибыль по регионам:", gross.sum(axis=1), "| маржа, %:", gross.sum(axis=1) / total * 100)

(4,) (4, 1) (4, 6) (4, 6)
Себестоимость:
 [[ 8400. 10500. 12600. 14700. 13300. 16100.]
 [ 6750.  8250. 10500. 12000.  9750. 11250.]
 [13000. 14300. 12350. 16250. 15600. 16900.]
 [ 6400. 10400. 13600. 12000. 11200. 14400.]]
Валовая прибыль:
 [[3600. 4500. 5400. 6300. 5700. 6900.]
 [2250. 2750. 3500. 4000. 3250. 3750.]
 [7000. 7700. 6650. 8750. 8400. 9100.]
 [1600. 2600. 3400. 3000. 2800. 3600.]]
Прибыль по регионам: [32400. 19500. 47600. 17000.] | маржа, %: [30. 25. 35. 20.]


**Вывод.** Валовая прибыль: Регион 3 — 47 600 (маржа 35%), Регион 1 — 32 400 (30%), Регион 2 — 19 500 (25%), Регион 4 — 17 000 (20%). Всего 116 500. Самая высокая выручка не всегда даёт самую высокую маржу: у Региона 4 выручка больше, чем у Региона 2, но прибыль меньше из-за высокой себестоимости.

## Задание 4. Сценарии скидок

Три сценария скидки `[0, 5%, 10%]` строятся одним выражением с broadcasting: массив `(3, 1, 1)` умножается на `(1, 4, 6)` и даёт `(3, 4, 6)` — по одной матрице 4×6 на сценарий. Объём продаж по условию не меняется, поэтому себестоимость остаётся прежней, а скидка уменьшает только выручку.

In [5]:
discount = np.array([0.0, 0.05, 0.10])
revenue_s = sales[np.newaxis, :, :] * (1 - discount)[:, np.newaxis, np.newaxis]
gross_s = revenue_s - cost[np.newaxis, :, :]
print(revenue_s.shape, gross_s.shape)
print("Выручка по сценариям:", revenue_s.sum(axis=(1, 2)))
print("Прибыль по сценариям:", gross_s.sum(axis=(1, 2)))
print("Прибыль по регионам и сценариям:\n", gross_s.sum(axis=2))
print("Изменение прибыли к базе, %:", (gross_s.sum(axis=(1, 2)) / gross_s[0].sum() - 1) * 100)

(3, 4, 6) (3, 4, 6)
Выручка по сценариям: [407000. 386650. 366300.]
Прибыль по сценариям: [116500.  96150.  75800.]
Прибыль по регионам и сценариям:
 [[32400. 19500. 47600. 17000.]
 [27000. 15600. 40800. 12750.]
 [21600. 11700. 34000.  8500.]]
Изменение прибыли к базе, %: [  0.   -17.47 -34.94]


**Вывод.** Выручка: 407 000 → 386 650 → 366 300. Валовая прибыль: 116 500 → 96 150 (−17,5%) → 75 800 (−34,9%). Скидка 5% стоит 20 350 — ровно 5% выручки, и вся она вычитается из прибыли, потому что себестоимость не меняется.

**Почему расчёт ненадёжен.** Мы предположили, что объём продаж не меняется. На практике скидка влияет на спрос (может увеличить его или лишь отдать скидку тем, кто купил бы и так), а себестоимость единицы и общая себестоимость зависят от объёма. Нет данных об эластичности спроса, реакции конкурентов, сезонности и повторных покупках, поэтому сценарий показывает только цену скидки при неизменном объёме, а не фактический результат кампании.

## Задание 5. Поиск финансового риска

Минимум по региону — `min(axis=1)` и `argmin`. Чувствительность к скидке — относительное падение прибыли при скидке 10% по сравнению с базой.

In [6]:
print("Минимальная прибыль по региону:", gross.min(axis=1), "в неделях", gross.argmin(axis=1) + 1)
print("Минимум в сценарии -10%:", gross_s[2].min(), "| отрицательных значений:", (gross_s < 0).sum())
loss = (1 - gross_s[2].sum(axis=1) / gross_s[0].sum(axis=1)) * 100
print("Падение прибыли при скидке 10%, % по регионам:", loss)
print("Самый чувствительный:", regions[loss.argmax()])

Минимальная прибыль по региону: [3600. 2250. 6650. 1600.] в неделях [1 1 3 1]
Минимум в сценарии -10%: 800.0 | отрицательных значений: 0
Падение прибыли при скидке 10%, % по регионам: [33.33 40.   28.57 50.  ]
Самый чувствительный: Регион 4


**Вывод.** Минимальная прибыль в базовом сценарии: Регион 4 — 1 600 (неделя 1), Регион 2 — 2 250 (неделя 1), Регион 1 — 3 600 (неделя 1), Регион 3 — 6 650 (неделя 3). Отрицательной прибыли нет даже при скидке 10% (минимум 800 — Регион 4, неделя 1), но запас прочности там минимальный. **Наиболее чувствителен к скидкам Регион 4**: при скидке 10% его прибыль падает на 50%, у Региона 2 — на 40%, у Региона 1 — на 33%, у Региона 3 — на 29%.